# An Overview of Soft Entropy Estimation with MNIST

We'll look at how you can estimate the entropy of some embeddings with just two lines of python. Or do online estimation of entropy and mutual estimation across batches of data using just 3 lines. Here we'll use the pytorch variant of the code, but there are backends for JAX and numpy that work identically.

We'll use the MNIST digits dataset from sklearn (64-dimensional pixel features, 10 digit classes) as a stand-in for model representations. The goal is to give an intuition for how the estimator behaves on real data with known structure:

- Raw pixel features should have **high entropy** (varied) and **moderate MI** with labels
- PCA dimensions ordered by variance: **high-variance PCs should carry more label MI** than low-variance ones
- Shuffling labels should **collapse MI to ≈ 0**
- Adding more classes should **increase entropy** of the representation

In [1]:
import torch
import numpy as np
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

from soft_entropy.pytorch import soft_entropy, soft_mutual_information

# load digits: 1797 samples, 64-dimensional pixel features, 10 classes
digits = load_digits()
z_np = digits.data.astype(np.float32)
y_np = digits.target

z = torch.tensor(z_np)
y = torch.tensor(y_np).long()

print(f"Dataset: {z.shape[0]} samples, {z.shape[1]} dimensions, {y.unique().numel()} classes")

Dataset: 1797 samples, 64 dimensions, 10 classes


## 1. Baseline: entropy and MI on raw pixel features

### with the setup above estimation of mutual information with respect to each class label across all the MNIST digits takes 2 lines of code and a fraction of a second.


In [2]:
h  = soft_entropy(z)
mi = soft_mutual_information(z, y)

#___________________print results_________________________
print(f"H(Z)        = {h:.4f}")
print(f"I(X;Z)      = {mi:.4f}")
print(f"I(X;Z)/H(Z) = {mi/h:.4f}")
print()

h_val = h.item()
per_digit_reg = {}
print(f"  {'digit':>5}  {'H(Z|X=x)':>10}  {'H(Z)-H(Z|x)':>13}  {'(H(Z)-H(Z|x))/H(Z)':>20}")
print("  " + "-" * 55)
for cls in range(10):
    mask_cls = torch.tensor(y_np == cls)
    h_cls    = soft_entropy(z[mask_cls]).item()
    mi_cls   = h_val - h_cls
    reg_cls  = mi_cls / h_val
    per_digit_reg[cls] = reg_cls
    print(f"  {cls:>5}  {h_cls:>10.4f}  {mi_cls:>13.4f}  {reg_cls:>20.4f}")


H(Z)        = 0.4104
I(X;Z)      = 0.0557
I(X;Z)/H(Z) = 0.1358

  digit    H(Z|X=x)    H(Z)-H(Z|x)    (H(Z)-H(Z|x))/H(Z)
  -------------------------------------------------------
      0      0.2390         0.1714                0.4177
      1      0.3180         0.0924                0.2251
      2      0.3306         0.0799                0.1946
      3      0.3529         0.0575                0.1400
      4      0.3637         0.0467                0.1138
      5      0.3815         0.0289                0.0704
      6      0.4094         0.0010                0.0024
      7      0.4240        -0.0136               -0.0331
      8      0.3843         0.0261                0.0636
      9      0.3430         0.0675                0.1644


![](figures/mnist_baseline.png)

The scores reflect how tightly each digit's embeddings cluster on the sphere relative to the full distribution. Lower H(Z|X=x) means the class occupies a more concentrated region; higher means it is spread broadly.

- **0 — highest regularity (0.42).** The oval is the most morphologically consistent digit in 8×8 pixels. Almost every 0 looks the same, so its embeddings cluster tightly and H(Z|X=0) is much lower than H(Z).

- **1, 2 — high regularity.** Simple strokes with relatively little writer variation, though more than 0.

- **3, 9 — mid-range.** Moderate shape variation (closed vs open loops, varying proportions).

- **5, 6 — low positive regularity.** Written with enough variability that the class embeddings nearly match the spread of the full distribution.

- **7 — negative regularity (−0.03).** Two things combine. First, 7 has genuinely high within-class shape variability — steep vs shallow angle, presence or absence of a crossbar, frequent visual overlap with 1. Second, there is a structural reason H(Z|X=x) can exceed H(Z): H(Z) is the entropy of the *average* assignment over all 1797 samples, and it is pulled down by very regular classes like 0 that concentrate mass in specific bins. A class like 7 that spreads its mass more evenly across the bins can therefore have higher entropy than this aggregate. The negativity is not an estimation artefact — it means 7s are genuinely more uniformly distributed across the embedding space than the dataset as a whole.

## 1.5. Online Estimation with the Accumulator

Repeats Example 1 using `SoftEntropyAccumulator` to stream the dataset in mini-batches of 12 rather than passing all embeddings at once. This allows passing batched data into the accumulator batch by batch then getting estimates for all the data. The code is very similar to before and just adds a line to instantiate the estimator before one line to pass the batch of data to the estimator. Note that this will automatically put the data onto any available GPUs.

In [3]:
from soft_entropy import SoftEntropyAccumulator

BATCH_SIZE = 12

acc = SoftEntropyAccumulator(d=z.shape[1], backend="torch")
for i in range(0, len(z), BATCH_SIZE):
    acc.update(z[i:i + BATCH_SIZE], labels=y[i:i + BATCH_SIZE])

h_acc  = acc.entropy()
mi_acc = acc.mutual_information()["labels"]
h_cond = acc.conditional_entropy()["labels"]   # {digit -> H(Z|X=digit)}

per_digit_reg_acc = {cls: (h_acc - h_cond[cls]) / h_acc for cls in range(10)}


#___________________print results_________________________

print(f"H(Z)        = {h_acc:.4f}   (full-batch: {h.item():.4f})")
print(f"I(X;Z)      = {mi_acc:.4f}   (full-batch: {mi.item():.4f})")
print(f"I(X;Z)/H(Z) = {mi_acc/h_acc:.4f}   (full-batch: {(mi/h).item():.4f})")
print()
print(f"  {'digit':>5}  {'H(Z|X=x)':>10}  {'H(Z)-H(Z|x)':>13}  {'(H(Z)-H(Z|x))/H(Z)':>20}")
print("  " + "-" * 55)
for cls in range(10):
    mi_cls  = h_acc - h_cond[cls]
    reg_cls = per_digit_reg_acc[cls]
    print(f"  {cls:>5}  {h_cond[cls]:>10.4f}  {mi_cls:>13.4f}  {reg_cls:>20.4f}")

H(Z)        = 0.6388   (full-batch: 0.4104)
I(X;Z)      = 0.0848   (full-batch: 0.0557)
I(X;Z)/H(Z) = 0.1327   (full-batch: 0.1358)

  digit    H(Z|X=x)    H(Z)-H(Z|x)    (H(Z)-H(Z|x))/H(Z)
  -------------------------------------------------------
      0      0.4455         0.1933                0.3026
      1      0.5060         0.1328                0.2079
      2      0.4540         0.1848                0.2893
      3      0.6555        -0.0167               -0.0261
      4      0.6188         0.0200                0.0313
      5      0.6385         0.0003                0.0004
      6      0.6142         0.0246                0.0385
      7      0.5830         0.0558                0.0874
      8      0.4416         0.1972                0.3087
      9      0.5739         0.0649                0.1015


## 2. Shuffled labels → MI ≈ 0

Randomly permuting labels breaks the relationship between embeddings and classes. MI should collapse.

![](figures/mnist_shuffled.png)

Randomly permuting the labels severs the relationship between embedding geometry and class identity. I(X;Z) collapses from 0.056 to 0.001 — close to zero but not exactly, since random permutations can produce spurious alignment by chance across a finite sample. The near-zero result confirms the estimator is measuring genuine geometric structure, not some artifact of the embedding distribution itself.

In [4]:
torch.manual_seed(42)
y_shuffled = y[torch.randperm(len(y))]

mi_shuffled = soft_mutual_information(z, y_shuffled)
print(f"I(X;Z) with correct labels  = {mi:.4f}")
print(f"I(X;Z) with shuffled labels = {mi_shuffled:.4f}")


I(X;Z) with correct labels  = 0.0557
I(X;Z) with shuffled labels = 0.0013


## 3. PCA dimensions — high-variance PCs carry more label MI

PCA orders components by explained variance. The leading PCs capture the most structure in the data and should therefore carry more information about digit class than the trailing PCs.

![](figures/mnist_pca.png)

PCA orders components by explained variance, so if the estimator is working correctly, MI should track variance across chunks. It does. The leading PCs (0–8) explain 67% of variance and carry an I(X;Z) of 0.20 and regularity of 0.215 — the digit class signal is heavily concentrated in the first few components. Each successive chunk drops in all three quantities.

In [5]:
pca = PCA(n_components=32)
z_pca = pca.fit_transform(z_np)  # [n, 32] — all PCs

chunk = 8
results = []
for start in range(0, 32, chunk):
    z_chunk = torch.tensor(z_pca[:, start:start + chunk].astype(np.float32))
    h_chunk  = soft_entropy(z_chunk).item()
    mi_chunk = soft_mutual_information(z_chunk, y).item()
    var_explained = pca.explained_variance_ratio_[start:start + chunk].sum()
    reg_chunk = mi_chunk / h_chunk if h_chunk > 0 else 0.0
    results.append((start, start + chunk, var_explained, h_chunk, mi_chunk, reg_chunk))

print(f"{'PCs':>10}  {'var explained':>14}  {'H(Z)':>8}  {'I(X;Z)':>8}  {'I(X;Z)/H(Z)':>13}")
print("-" * 62)
for start, end, var, h_c, mi_c, reg_c in results:
    print(f"  {start:2d}–{end:2d}      {var:>14.3f}  {h_c:>8.4f}  {mi_c:>8.4f}  {reg_c:>13.4f}")


       PCs   var explained      H(Z)    I(X;Z)    I(X;Z)/H(Z)
--------------------------------------------------------------
   0– 8               0.674    0.9742    0.2092         0.2147
   8–16               0.175    0.9893    0.0749         0.0757
  16–24               0.077    0.9923    0.0350         0.0353
  24–32               0.040    0.9923    0.0280         0.0282


## 4. More classes → higher entropy

Including more digit classes exposes more diversity in the representation space. Entropy should rise as we add classes.

![](figures/mnist_n_classes.png)

Adding digit classes increases both H(Z) and I(X;Z) as the representation space becomes more diverse and more label structure becomes available. The relationship between number of classes and mutual information isn't quite monotone (though it's close!), like because some classes are actually more spread out than P(Z).

In [6]:
print(f"{'n_classes':>10}  {'H(Z)':>8}  {'I(X;Z)':>8}  {'I(X;Z)/H(Z)':>13}")
print("-" * 46)
n_classes_range = list(range(2, 11))
entropies, mis, regs = [], [], []
for n_classes in n_classes_range:
    mask = y < n_classes
    z_sub = z[mask]
    y_sub = y[mask]
    h_sub  = soft_entropy(z_sub).item()
    mi_sub = soft_mutual_information(z_sub, y_sub).item()
    reg_sub = mi_sub / h_sub if h_sub > 0 else 0.0
    entropies.append(h_sub)
    mis.append(mi_sub)
    regs.append(reg_sub)
    print(f"{n_classes:>10}  {h_sub:>8.4f}  {mi_sub:>8.4f}  {reg_sub:>13.4f}")

is_monotone = all(entropies[i] <= entropies[i+1] for i in range(len(entropies)-1))
print(f"\nEntropy monotonically increasing with n_classes: {is_monotone}")



 n_classes      H(Z)    I(X;Z)    I(X;Z)/H(Z)
----------------------------------------------
         2    0.2903    0.0113         0.0390
         3    0.3147    0.0187         0.0595
         4    0.3286    0.0181         0.0552
         5    0.3468    0.0257         0.0740
         6    0.3566    0.0253         0.0709
         7    0.3882    0.0457         0.1178
         8    0.4157    0.0631         0.1518
         9    0.4138    0.0578         0.1397
        10    0.4104    0.0557         0.1358

Entropy monotonically increasing with n_classes: False
